# FindMaximumSubarray and FindMaxCrossingSubarray (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/FindMaximumSubarray_java.ipynb)

Chapter 17 — Recursive algorithms. Runnable notebook of `FindMaximumSubarray` and `FindMaxCrossingSubarray`: the algorithms in Java, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/17-recursive-algorithms/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/FindMaximumSubarray_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/FindMaximumSubarray_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `recursiveSum` (camelCase).

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## FindMaxCrossingSubarray and FindMaximumSubarray (divide and conquer)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: an array and three indices $l\le mid<r$. *Output*: the indices $maxleft\le mid<maxright$ of a maximum-sum subarray among those crossing the midpoint, and its sum. It scans the left half from $mid$ down to $l$ and the right half from $mid+1$ to $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the indices $i\le j$ of a maximum-sum subarray of $a[l..r]$ and its sum. The best among the maximum subarray of the left half, of the right half and of those crossing the midpoint (in case of ties, the left one first, then the right one). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ calls.

The $-\infty$ of the pseudo-code is the smallest representable value. First the trace on the example $a=[-2,5,-1,3]$ (like the execution of the C++ implementation of the chapter; result $(1,3,7)$, that is $(2,4,7)$ with indices from 1), then the stock example, with the daily changes $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: table of the results of the calls on the sub-intervals (by level of the tree, as in the chapter; indices are from 0) and the computations of the two `FindMaxCrossingSubarray` of level 1. The result is $(7,10,43)$, that is $(8,11,43)$ with indices from 1.

In [2]:
%%writefile MaxSubDC.java
import java.util.*;

public class MaxSubDC {
    static int fmsCalls = 0;   // calls of FindMaximumSubarray

    static String tr(int[] t) {
        return "(" + t[0] + ", " + t[1] + ", " + t[2] + ")";
    }

    // Returns {maxleft, maxright, sum} of the best subarray crossing mid
    static int[] findMaxCrossingSubarray(int[] a, int l, int mid, int r) {
        int leftsum = Integer.MIN_VALUE;        // -infinity
        int sum = 0;
        int maxleft = mid;
        for (int i = mid; i >= l; i--) {        // from mid down to l
            sum = sum + a[i];                   // sum of a[i..mid]
            if (sum > leftsum) {
                leftsum = sum;
                maxleft = i;
            }
        }
        int rightsum = Integer.MIN_VALUE;
        sum = 0;
        int maxright = mid + 1;
        for (int j = mid + 1; j <= r; j++) {    // from mid+1 to r
            sum = sum + a[j];                   // sum of a[mid+1..j]
            if (sum > rightsum) {
                rightsum = sum;
                maxright = j;
            }
        }
        return new int[]{maxleft, maxright, leftsum + rightsum};
    }

    // Returns {i, j, sum} of a maximum-sum subarray of a[l..r]
    static int[] findMaximumSubarray(int[] a, int l, int r) {
        fmsCalls++;
        if (l == r) {
            return new int[]{l, r, a[l]};       // base case: one element
        }
        int mid = l + (r - l) / 2;
        int[] left = findMaximumSubarray(a, l, mid);
        int[] right = findMaximumSubarray(a, mid + 1, r);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        if (left[2] >= right[2] && left[2] >= cross[2]) {
            return left;
        }
        if (right[2] >= left[2] && right[2] >= cross[2]) {
            return right;
        }
        return cross;
    }

    // ---- variants that print the trace ----
    static int[] best(int[] left, int[] right, int[] cross) {
        if (left[2] >= right[2] && left[2] >= cross[2]) return left;
        if (right[2] >= left[2] && right[2] >= cross[2]) return right;
        return cross;
    }

    static int[] fmsTrace(int[] a, int l, int r, int depth) {
        String pad = " ".repeat(2 * depth);
        if (l == r) {
            System.out.printf("%sFindMaximumSubarray(a, %d, %d) = (%d, %d, %d)  base case%n", pad, l, r, l, r, a[l]);
            return new int[]{l, r, a[l]};
        }
        int mid = l + (r - l) / 2;
        System.out.printf("%sFindMaximumSubarray(a, %d, %d): l != r, mid = %d%n", pad, l, r, mid);
        int[] left = fmsTrace(a, l, mid, depth + 1);
        int[] right = fmsTrace(a, mid + 1, r, depth + 1);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        System.out.printf("%s  FindMaxCrossingSubarray(a, %d, %d, %d) = %s%n", pad, l, mid, r, tr(cross));
        int[] ret = best(left, right, cross);
        System.out.printf("%s  left.sum=%d, right.sum=%d, cross.sum=%d => returns %s%n", pad, left[2], right[2], cross[2], tr(ret));
        return ret;
    }

    // each row: {l, r, mid} + left + right + cross + ret
    static int[] fmsRows(int[] a, int l, int r, ArrayList<int[][]> rows) {
        if (l == r) {
            return new int[]{l, r, a[l]};
        }
        int mid = l + (r - l) / 2;
        int[] left = fmsRows(a, l, mid, rows);
        int[] right = fmsRows(a, mid + 1, r, rows);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        int[] ret = best(left, right, cross);
        rows.add(new int[][]{{l, r, mid}, left, right, cross, ret});
        return ret;
    }

    static void crossingTrace(int[] a, int l, int mid, int r) {
        System.out.printf("FindMaxCrossingSubarray(a, %d, %d, %d)%n", l, mid, r);
        System.out.printf("%3s %5s %5s %8s %8s%n", "i", "a[i]", "sum", "leftsum", "maxleft");
        int leftsum = Integer.MIN_VALUE, maxleft = mid, sum = 0;
        for (int i = mid; i >= l; i--) {
            sum = sum + a[i];
            if (sum > leftsum) {
                leftsum = sum;
                maxleft = i;
            }
            System.out.printf("%3d %5d %5d %8d %8d%n", i, a[i], sum, leftsum, maxleft);
        }
        System.out.printf("%3s %5s %5s %8s %8s%n", "j", "a[j]", "sum", "rightsum", "maxright");
        int rightsum = Integer.MIN_VALUE, maxright = mid + 1;
        sum = 0;
        for (int j = mid + 1; j <= r; j++) {
            sum = sum + a[j];
            if (sum > rightsum) {
                rightsum = sum;
                maxright = j;
            }
            System.out.printf("%3d %5d %5d %8d %8d%n", j, a[j], sum, rightsum, maxright);
        }
        System.out.printf("crossing subarray a[%d..%d], sum %d + %d = %d%n", maxleft, maxright, leftsum, rightsum, leftsum + rightsum);
    }

    public static void main(String[] args) {
        int[] a = {-2, 5, -1, 3};
        System.out.println("a = " + Arrays.toString(a));
        int[] res = fmsTrace(a, 0, 3, 0);
        fmsCalls = 0;
        findMaximumSubarray(a, 0, 3);
        System.out.printf("result %s   (%d, %d, %d with indices from 1)   calls=%d (2n-1=%d)%n", tr(res), res[0] + 1, res[1] + 1, res[2], fmsCalls, 2 * 4 - 1);

        System.out.println();
        int[] b = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
        int n = 16;
        System.out.println("a = " + Arrays.toString(b));
        fmsCalls = 0;
        res = findMaximumSubarray(b, 0, n - 1);
        System.out.printf("FindMaximumSubarray(a, 0, %d) = %s   (%d, %d, %d with indices from 1)   calls=%d (2n-1=%d)%n", n - 1, tr(res), res[0] + 1, res[1] + 1, res[2], fmsCalls, 2 * n - 1);
        ArrayList<int[][]> rows = new ArrayList<>();
        fmsRows(b, 0, n - 1, rows);
        rows.sort((x, y) -> (x[0][1] - x[0][0]) != (y[0][1] - y[0][0]) ? (y[0][1] - y[0][0]) - (x[0][1] - x[0][0]) : x[0][0] - y[0][0]);
        System.out.printf("%-9s %3s  %-13s %-13s %-13s returned%n", "call", "mid", "left", "right", "crossing");
        for (int[][] w : rows) {
            String call = "[" + w[0][0] + ".." + w[0][1] + "]";
            System.out.printf("%-9s %3d  %-13s %-13s %-13s %s%n", call, w[0][2], tr(w[1]), tr(w[2]), tr(w[3]), tr(w[4]));
        }
        System.out.println();
        crossingTrace(b, 0, 3, 7);
        System.out.println();
        crossingTrace(b, 8, 11, 15);
    }
}

Writing MaxSubDC.java


In [3]:
!javac MaxSubDC.java && java MaxSubDC

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  base case
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  base case
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => returns (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  base case
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  base case
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => returns (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => returns (1, 3, 7)
result (1, 3, 7)   (2, 4, 7 with indices from 1)   calls=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 with indices from 1)   calls=31 (2n-1=31)
call      mid  left          righ

**Try it yourself.** Change the array: the maximum subarray may have a single element or cross the midpoint; the calls are always $2n-1$.

In [4]:
%%writefile ProvaMaxSubDC.java
import java.util.*;

public class ProvaMaxSubDC {
    public static void main(String[] args) {
        // Change the array: how many elements does the maximum subarray have? The calls are always 2n-1.
        int[] b = {4, -1, 2, 1, -5, 4, -7, 3, 3};
        MaxSubDC.fmsCalls = 0;
        int[] t = MaxSubDC.findMaximumSubarray(b, 0, b.length - 1);
        System.out.println(MaxSubDC.tr(t) + " " + MaxSubDC.fmsCalls);
    }
}

Writing ProvaMaxSubDC.java


In [5]:
!javac ProvaMaxSubDC.java && java ProvaMaxSubDC

(0, 3, 6) 17


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).